# Advanced Modeling for Skewed Target (Movie Gross Revenue)
This notebook handles highly skewed targets by implementing baseline, log-transformed, and segmented modeling approaches.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# --- 1. DATA SETUP & PREPROCESSING ---
df = pd.read_csv('../datasets/movies.csv')
df = df.dropna(subset=['budget', 'gross'])
if 'rating' in df.columns:
    df['rating'] = df['rating'].fillna('Unrated')
df = df.dropna()
df['budget'] = df['budget'].astype('int64')
df['gross'] = df['gross'].astype('int64')
df['year_correct'] = df['released'].astype(str).str.extract(r'(\d{4})')
df['year_correct'] = pd.to_numeric(df['year_correct'], errors='coerce')
df = df.dropna(subset=['year_correct'])

features = ['budget', 'score', 'votes', 'runtime', 'year_correct', 'rating', 'genre', 'director', 'writer']
target = 'gross'

X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Encode Low Cardinality
for col in ['rating', 'genre']:
    encoder = LabelEncoder()
    encoder.fit(X_train[col].astype(str))
    classes = list(encoder.classes_)
    if '<UNSEEN>' not in classes: classes.append('<UNSEEN>')
    encoder.classes_ = np.array(classes)
    
    X_train[col] = encoder.transform(X_train[col].astype(str))
    known_classes = set(encoder.classes_)
    X_test[col] = X_test[col].astype(str).apply(lambda x: x if x in known_classes else '<UNSEEN>')
    X_test[col] = encoder.transform(X_test[col])

# Target Encode High Cardinality
global_mean = y_train.mean()
for col in ['director', 'writer']:
    target_map = y_train.groupby(X_train[col]).mean().to_dict()
    X_train[col] = X_train[col].map(target_map).fillna(global_mean)
    X_test[col] = X_test[col].map(target_map).fillna(global_mean)

print(f"Data Prep Complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Data Prep Complete. Train shape: (4345, 9), Test shape: (1087, 9)


## Evaluation Helper

In [2]:
# --- EVALUATION HELPER FUNCTION ---
def evaluate_model(y_true, y_pred, model_name):
    """Computes and prints regression metrics."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"R² Score : {r2:.4f}")
    print(f"RMSE     : ${rmse:,.2f}")
    print(f"MAE      : ${mae:,.2f}\n")
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae}

In [3]:
# --- 2. BASELINE MODEL ---
baseline_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)
y_pred_base = baseline_model.predict(X_test)

metrics_base = evaluate_model(y_test, y_pred_base, "Baseline Model (Raw Target)")

--- Baseline Model (Raw Target) ---
R² Score : 0.5785
RMSE     : $120,272,675.32
MAE      : $62,264,613.02



In [4]:
# --- 3. HANDLE SKEWNESS (LOG TRANSFORMATION) ---
# Apply log1p (log(1 + x)) to handle large skewed values safely
y_train_log = np.log1p(y_train)

log_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
log_model.fit(X_train, y_train_log)

# Predict in log scale, then convert back using expm1
y_pred_log_scale = log_model.predict(X_test)
y_pred_log_transformed = np.expm1(y_pred_log_scale)

metrics_log = evaluate_model(y_test, y_pred_log_transformed, "Log-Transformed Target Model")

print("Insight: Log transformation reduces the impact of extreme outlier blockbusters during training, which can stabilize MAE.")

--- Log-Transformed Target Model ---
R² Score : 0.5434
RMSE     : $125,181,808.12
MAE      : $56,809,910.09

Insight: Log transformation reduces the impact of extreme outlier blockbusters during training, which can stabilize MAE.


In [5]:
# --- 4. ADVANCED SEGMENTED MODELING & TWO-STAGE APPROACH ---
import numpy as np
import pandas as pd

# Load advanced estimators (fallback to sklearn GradientBoosting if XGBoost is not installed)
try:
    from xgboost import XGBRegressor, XGBClassifier
    has_xgb = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor
    from sklearn.ensemble import GradientBoostingClassifier as XGBClassifier
    has_xgb = False

# STEP 1: Quantile-based Segmentation (Balanced Classes)
def segment_target_quantile(y_train, y_test, num_segments=3):
    """Creates robust quantile-based segments to ensure balanced class distributions."""
    # pd.qcut creates equal-sized bins based on empirical quantiles
    y_train_seg, bins = pd.qcut(y_train, q=num_segments, labels=False, retbins=True)
    
    # Apply the same bins to the test set. Fillna handles any unseen edge cases (extrapolating min/max).
    bins[0], bins[-1] = -np.inf, np.inf 
    y_test_seg = pd.cut(y_test, bins=bins, labels=False, include_lowest=True).fillna(0).astype(int)
    
    return y_train_seg, y_test_seg

y_train_segment, y_test_segment = segment_target_quantile(y_train, y_test, num_segments=3)

# STEP 2: Train Classifier for Soft-Probability Evaluation
print(f"Training Classifier to predict segments (Using {'XGBoost' if has_xgb else 'GradientBoosting'})...")
classifier = XGBClassifier(n_estimators=100, max_depth=5, random_state=42)
classifier.fit(X_train, y_train_segment)

# Generate soft prediction probabilities
segment_probs = classifier.predict_proba(X_test)

# STEP 3: Train Segment-Specific Regression Models with Log-Transformation
print("Training Segment-Specific Regression Models (Target: log1p) & Global Fallback...")
segment_models = {}

for seg in [0, 1, 2]:
    mask = (y_train_segment == seg)
    
    # Skip segments with very few samples (Edge Case Safety)
    if mask.sum() < 10:
        print(f"Warning: Segment {seg} has too few samples. Skipping.")
        continue
        
    X_seg, y_seg = X_train[mask], y_train[mask]
    
    # Log Transformation for target!
    y_seg_log = np.log1p(y_seg)
    
    # Train specialized models
    model = XGBRegressor(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_seg, y_seg_log)
    segment_models[seg] = model

# Baseline Fallback Model (For low-confidence classification cases)
global_fallback_model = XGBRegressor(n_estimators=150, max_depth=6, random_state=42)
global_fallback_model.fit(X_train, np.log1p(y_train))

# STEP 4: Soft Prediction with Baseline Fallback Mechanism
print("Generating Predictions (Weighted Sum Array)...")
y_pred_advanced = np.zeros(X_test.shape[0], dtype=float)

# Pre-calculate log-transformed predictions across all models to vectorize operations where possible
segment_preds_log = np.zeros((X_test.shape[0], 3))
for seg in segment_models:
    segment_preds_log[:, seg] = segment_models[seg].predict(X_test)

# Predict global fallback
global_preds_log = global_fallback_model.predict(X_test)

for i in range(X_test.shape[0]):
    probs = segment_probs[i]
    max_prob = np.max(probs)
    
    # Inverse Transformation (expm1)
    seg_preds = np.expm1(segment_preds_log[i])
    global_pred = np.expm1(global_preds_log[i])
    
    # Fallback Mechanism: If the model isn't at least 50% sure about a segment
    if max_prob < 0.5:
        y_pred_advanced[i] = global_pred
    else:
        # Predict using a weighted sum representation instead of a hard decision!
        y_pred_advanced[i] = np.dot(probs, seg_preds)

# Evaluate the advanced implementation
metrics_advanced = evaluate_model(y_test, y_pred_advanced, "Advanced Two-Stage (Weighted + Fallback)")

print("Insight: Using predict_proba() allows a smooth transition between segments, avoiding rigid boundaries.")
print("Insight: Using pd.qcut ensures balanced training bins, preventing any specific model from overfitting to limited samples.")
print("Insight: The global fallback ensures highly robust predictions when segment confidence is weak.")

Training Classifier to predict segments (Using GradientBoosting)...
Training Segment-Specific Regression Models (Target: log1p) & Global Fallback...
Generating Predictions (Weighted Sum Array)...
--- Advanced Two-Stage (Weighted + Fallback) ---
R² Score : 0.6127
RMSE     : $115,282,724.13
MAE      : $55,053,072.56

Insight: Using predict_proba() allows a smooth transition between segments, avoiding rigid boundaries.
Insight: Using pd.qcut ensures balanced training bins, preventing any specific model from overfitting to limited samples.
Insight: The global fallback ensures highly robust predictions when segment confidence is weak.


In [6]:
# --- 5. COMPARISON --- 
print("\n=== FINAL MODEL COMPARISON ===")
comparison_df = pd.DataFrame({
    'Baseline': metrics_base,
    'Log-Transformed': metrics_log,
    'Advanced Soft-Voting': metrics_advanced
}).T

display(comparison_df.style.format("{:,.4f}"))

# Recommendation
best_r2_model = comparison_df['R2'].idxmax()
best_mae_model = comparison_df['MAE'].idxmin()
print(f"\n🏆 Best Model for overall variance (R²): {best_r2_model}")
print(f"🏆 Best Model for absolute accuracy (MAE): {best_mae_model}")


=== FINAL MODEL COMPARISON ===


,R2,RMSE,MAE
Baseline,0.5785,"120,272,675.3169","62,264,613.0239"
Log-Transformed,0.5434,"125,181,808.1192","56,809,910.0866"
Advanced Soft-Voting,0.6127,"115,282,724.1333","55,053,072.5640"



🏆 Best Model for overall variance (R²): Advanced Soft-Voting
🏆 Best Model for absolute accuracy (MAE): Advanced Soft-Voting
